# FLAG 2027 - Notebook 11: Head & training tricks from the literature (+ ensemble test)

Ideas found by our literature search (FAME 2024/2026 system papers), tested on top of **bal75** (CV 26.89, dev 31.24), using features already on Drive:

| Version | Idea | Source |
|---|---|---|
| H1_fop_indrop | our FOP head + input dropout 0.5 on the frozen embeddings | FAME26 winner used input dropout 0.9 |
| H2_fop_ema | our FOP head + EMA weight averaging | XM-ALIGN / SWA |
| H3_fop_nce | our FOP head + cross-modal contrastive loss (InfoNCE, learnable temperature) | FAME24 #2 (SCC) |
| H4_lin_mse | **winner-style head**: one Linear layer per modality (192-d), input dropout 0.7, SHARED cosine classifier, MSE face-voice alignment, cosine scoring | FAME26 #1 + XM-ALIGN |
| H5_lin_nce | the same linear head with InfoNCE instead of MSE | |

Every run also saves its validation scores, so at the end we test **score-level ensembles** of the versions.

About 75-100 min. Resumable. Paste the **SUMMARY** to Claude.

In [ ]:
# Cell 1 - Drive + GPU
from google.colab import drive
drive.mount('/content/drive')
import torch
dev = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', dev)

In [ ]:
# Cell 2 - load challenge, MAV-Celeb and VoxCeleb2 features (as Notebooks 09/10)
import os, re, json, time, zipfile, copy
import numpy as np, pandas as pd

BASE = '/content/drive/MyDrive'
PROJECT = os.path.join(BASE, 'FLAG2027')
FEATDIR = os.path.join(PROJECT, 'features')
RESDIR = os.path.join(PROJECT, 'results')
SCOREDIR = os.path.join(RESDIR, 'nb11_scores'); os.makedirs(SCOREDIR, exist_ok=True)
ZIP_DIR = json.load(open(os.path.join(PROJECT, 'paths.json')))['ZIP_DIR']
LOCAL = '/content/flag_small'
os.makedirs(LOCAL, exist_ok=True)
for z in ['train_set.zip', 'dev_set.zip']:
    flag = os.path.join(LOCAL, z + '.done')
    if not os.path.exists(flag):
        with zipfile.ZipFile(os.path.join(ZIP_DIR, z)) as zf:
            zf.extractall(LOCAL, members=[m for m in zf.namelist() if m.endswith(('.csv', '.txt'))])
        open(flag, 'w').close()
INDEX = [os.path.join(r, f) for r, _, fs in os.walk(LOCAL) for f in fs if f.endswith(('.csv', '.txt'))]
def find_one(*parts):
    hits = [p for p in INDEX if all(x in p for x in parts)]
    assert len(hits) == 1, 'looking for %s -> found %s' % (parts, hits)
    return hits[0]
def read_feat(path, dim):
    df = pd.read_csv(path, header=None)
    return df.iloc[:, :dim].values.astype(np.float32), (df.iloc[:, dim].values if df.shape[1] > dim else None)

SPLIT_NAMES = ['train', 'no_gender_English', 'no_gender_Bangla', 'gender_English', 'gender_Bangla']
FEAT = {}
Xv, y = read_feat(find_one('train_English_voices.csv'), 192)
FEAT['ecapa'] = {'train': Xv}
for s in SPLIT_NAMES[1:]:
    track, lang = s.rsplit('_', 1)
    FEAT['ecapa'][s] = read_feat(find_one('/' + track + '/', lang + '_test_voices.csv'), 192)[0]
for name in ['clip', 'wavlm_sv']:
    d = np.load(os.path.join(FEATDIR, name + '.npz')); FEAT[name] = {s: d[s] for s in SPLIT_NAMES}
y = y.astype(int)
train_lines = [l.split() for l in open(find_one('train_English.txt')) if l.strip()]
cls2spk = {int(l[5]): l[4] for l in train_lines}
meta = pd.read_csv(os.path.join(ZIP_DIR, 'meta_file_train_set.csv'), header=None, names=['spk', 'g'])
spk2g = dict(zip(meta.spk.astype(str).str.strip(), meta.g.astype(str).str.strip().str.lower()))
ids = np.unique(y)
id_gender = {c: spk2g[cls2spk[c]] for c in ids}
MAV = dict(np.load(os.path.join(FEATDIR, 'mavceleb_v1v3.npz')))
VOXF = [dict(np.load(os.path.join(FEATDIR, f))) for f in sorted(os.listdir(FEATDIR)) if f.startswith('vox_') and f.endswith('.npz')]
VOX = {k: np.concatenate([v[k] for v in VOXF]) for k in ['spk', 'clip', 'ecapa', 'wavlm_sv']}
SUMMARY = []
def note(msg):
    print(msg); SUMMARY.append(msg)
print('challenge', len(y), '| MAV', len(MAV['spk']), '| VOX', len(VOX['spk']))

In [ ]:
# Cell 3 - combined table, per-dataset normalization, validation folds (same as before)
import torch.nn as nn
import torch.nn.functional as Fn
from sklearn.metrics import roc_curve

def l2(X):
    return X / (np.linalg.norm(X, axis=1, keepdims=True) + 1e-9)
def face_of(clip):
    return l2(clip).astype(np.float32)
def voice_of(ecapa, wavlm):
    return np.concatenate([l2(ecapa), l2(wavlm)], 1).astype(np.float32)

N_CH, N_MAV, N_VOX = len(y), len(MAV['spk']), len(VOX['spk'])
ext_spk = list(MAV['spk']) + list(VOX['spk'])
ext_lab = {s: 1000 + i for i, s in enumerate(sorted(set(ext_spk)))}
Y_ALL = np.concatenate([y, np.array([ext_lab[s] for s in ext_spk])])
D_ALL = np.r_[np.zeros(N_CH, int), np.ones(N_MAV, int), 2 * np.ones(N_VOX, int)]
F_RAW = np.concatenate([face_of(FEAT['clip']['train']), face_of(MAV['clip']), face_of(VOX['clip'])])
V_RAW = np.concatenate([voice_of(FEAT['ecapa']['train'], FEAT['wavlm_sv']['train']), voice_of(MAV['ecapa'], MAV['wavlm_sv']), voice_of(VOX['ecapa'], VOX['wavlm_sv'])])
EXT = N_CH + np.arange(N_MAV + N_VOX)

def domain_normalize(tr_idx):
    F, V = F_RAW.copy(), V_RAW.copy(); ch_stats = None
    for d in np.unique(D_ALL[tr_idx]):
        trd = tr_idx[D_ALL[tr_idx] == d]; alld = np.where(D_ALL == d)[0]
        st = (F_RAW[trd].mean(0), F_RAW[trd].std(0) + 1e-6, V_RAW[trd].mean(0), V_RAW[trd].std(0) + 1e-6)
        F[alld] = (F_RAW[alld] - st[0]) / st[1]; V[alld] = (V_RAW[alld] - st[2]) / st[3]
        if d == 0:
            ch_stats = st
    return F, V, ch_stats

def eer(labels, scores):
    fpr, tpr, _ = roc_curve(labels, scores); fnr = 1 - tpr
    i = np.nanargmin(np.abs(fnr - fpr)); return (fpr[i] + fnr[i]) / 2 * 100
def make_pairs(idx, n=2000, same_gender_neg=False, seed=0):
    rng = np.random.default_rng(seed)
    spk_ = np.unique(y[idx]); rows_ = {s: idx[y[idx] == s] for s in spk_}
    V, F, L = [], [], []
    for _ in range(n):
        s = rng.choice(spk_); V.append(rng.choice(rows_[s])); F.append(rng.choice(rows_[s])); L.append(1)
        s = rng.choice(spk_)
        cand = [t for t in spk_ if t != s and (not same_gender_neg or id_gender[t] == id_gender[s])] or [t for t in spk_ if t != s]
        t = rng.choice(cand); V.append(rng.choice(rows_[s])); F.append(rng.choice(rows_[t])); L.append(0)
    return np.array(V), np.array(F), np.array(L)
K = 5
rng = np.random.default_rng(42)
folds = [[] for _ in range(K)]
for gg in sorted(set(id_gender.values())):
    members = [c for c in ids if id_gender[c] == gg]; rng.shuffle(members)
    for i, c in enumerate(members):
        folds[i % K].append(c)
FOLD_DATA = []
for k in range(K):
    val_ids = set(folds[k])
    va_idx = np.where([c in val_ids for c in y])[0]; tr_idx = np.where([c not in val_ids for c in y])[0]
    FOLD_DATA.append((tr_idx, [make_pairs(va_idx, seed=k), make_pairs(va_idx, same_gender_neg=True, seed=100 + k)]))
print('ready')

In [ ]:
# Cell 4 - flexible head: 'fop' (ours) or 'lin' (winner-style), optional input dropout, InfoNCE, MSE, EMA
class Branch(nn.Module):
    def __init__(self, d_in, d=128, in_drop=0.0):
        super().__init__()
        self.net = nn.Sequential(nn.Dropout(in_drop), nn.Linear(d_in, d), nn.BatchNorm1d(d), nn.ReLU(), nn.Dropout(0.5))
    def forward(self, x):
        return self.net(x)

class Head(nn.Module):
    def __init__(self, n_class, d_face, d_voice, arch='fop', d=128, in_drop=0.0):
        super().__init__()
        self.arch = arch
        if arch == 'fop':
            self.face, self.voice = Branch(d_face, d, in_drop), Branch(d_voice, d, in_drop)
            self.att = nn.Sequential(nn.Linear(2 * d, d), nn.BatchNorm1d(d), nn.ReLU(), nn.Linear(d, d))
            self.cls = nn.Linear(d, n_class)
        else:   # winner-style: one linear layer per modality + shared cosine classifier
            self.face = nn.Sequential(nn.Dropout(in_drop), nn.Linear(d_face, d))
            self.voice = nn.Sequential(nn.Dropout(in_drop), nn.Linear(d_voice, d))
            self.W = nn.Parameter(torch.randn(n_class, d) * 0.01)
        self.log_t = nn.Parameter(torch.tensor(np.log(0.07), dtype=torch.float32))   # InfoNCE temperature
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight); nn.init.constant_(m.bias, 0.01)
    def embed(self, xf, xv):
        return Fn.normalize(self.face(xf), dim=1), Fn.normalize(self.voice(xv), dim=1)
    def score(self, xf, xv):   # HIGHER = same person
        f, v = self.embed(xf, xv)
        if self.arch == 'fop':
            return -(torch.tanh(f) - torch.tanh(v)).norm(dim=1)
        return (f * v).sum(1)

def opl_loss(feat, lab):
    feat = Fn.normalize(feat, dim=1); dot = feat @ feat.t()
    same = (lab[:, None] == lab[None, :]).float()
    pos = same - torch.eye(len(lab), device=feat.device); neg = 1 - same
    return (1 - (pos * dot).sum() / (pos.sum() + 1e-6)) + 0.7 * (neg * dot.abs()).sum() / (neg.sum() + 1e-6)
def nce_loss(f, v, lab, log_t):
    # supervised cross-modal InfoNCE: every face of the same person is a positive for a voice (and vice versa)
    sim = v @ f.t() / log_t.exp().clamp(min=0.01)
    pos = (lab[:, None] == lab[None, :]).float()
    l1 = -(pos * sim.log_softmax(1)).sum(1) / pos.sum(1)
    l2_ = -(pos.t() * sim.t().log_softmax(1)).sum(1) / pos.t().sum(1)
    return (l1.mean() + l2_.mean()) / 2

def total_loss(model, xf, xv, lab, cfg):
    f, v = model.embed(xf, xv)
    if model.arch == 'fop':
        ft, vt = torch.tanh(f), torch.tanh(v)
        a = torch.sigmoid(model.att(torch.cat([f, v], 1)))
        fused = ft * a + (1 - a) * vt
        loss = Fn.cross_entropy(model.cls(fused), lab) + 2.0 * opl_loss(fused, lab)
    else:
        Wn = Fn.normalize(model.W, dim=1)
        loss = (Fn.cross_entropy(16 * f @ Wn.t(), lab) + Fn.cross_entropy(16 * v @ Wn.t(), lab)) / 2
        loss = loss + opl_loss(torch.cat([f, v]), torch.cat([lab, lab]))
    if cfg.get('mse', 0) > 0:
        loss = loss + cfg['mse'] * ((f - v) ** 2).sum(1).mean()
    if cfg.get('nce', 0) > 0:
        loss = loss + cfg['nce'] * nce_loss(f, v, lab, model.log_t)
    return loss
print('heads ready')

In [ ]:
# Cell 5 - training (bal75 data recipe) + run all versions (RESUMABLE; validation scores saved for ensembling)
SEEDS = [1, 2, 3]
P_ID, K_CLIP, LR, MAX_EP, PATIENCE, STEPS, IN_FRAC = 16, 8, 1e-4, 100, 15, 60, 0.75

@torch.no_grad()
def val_scores(model, XF, XV, vsets):
    model.eval()
    return [model.score((XF[Fi] - model.mf) / model.sf, (XV[Vi] - model.mv) / model.sv).cpu().numpy() for Vi, Fi, L in vsets]

def train_variant(tr_idx, cfg, vsets=None, seed=1, max_epochs=MAX_EP):
    torch.manual_seed(seed); rng = np.random.default_rng(seed)
    all_tr = np.concatenate([tr_idx, EXT])
    F, V, ch_stats = domain_normalize(all_tr)
    XF = torch.tensor(F, device=dev); XV = torch.tensor(V, device=dev)
    cl = np.unique(Y_ALL[all_tr]); rm = {c: i for i, c in enumerate(cl)}
    ytr_np = np.array([rm[c] for c in Y_ALL[all_tr]]); ytr = torch.tensor(ytr_np, device=dev)
    cls_in = np.zeros(len(cl), bool); cls_in[np.unique(ytr_np[D_ALL[all_tr] < 2])] = True
    in_cls, out_cls = np.where(cls_in)[0], np.where(~cls_in)[0]; n_in = int(round(P_ID * IN_FRAC))
    pos_by_cls = [np.where(ytr_np == c)[0] for c in range(len(cl))]
    tr = torch.tensor(all_tr, device=dev)
    model = Head(len(cl), XF.shape[1], XV.shape[1], cfg['arch'], cfg.get('d', 128), cfg.get('in_drop', 0.0)).to(dev)
    model.mf, model.sf, model.mv, model.sv = XF[tr].mean(0), XF[tr].std(0) + 1e-6, XV[tr].mean(0), XV[tr].std(0) + 1e-6
    opt = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=0.01)
    ema = copy.deepcopy(model) if cfg.get('ema') else None
    best, best_ep, best_state, wait = 1e9, max_epochs, None, 0
    for ep in range(1, max_epochs + 1):
        model.train()
        for _ in range(STEPS):
            chosen = np.r_[rng.choice(in_cls, n_in, replace=False), rng.choice(out_cls, P_ID - n_in, replace=False)]
            j = torch.tensor(np.concatenate([rng.choice(pos_by_cls[c], K_CLIP, replace=len(pos_by_cls[c]) < K_CLIP) for c in chosen]), device=dev)
            loss = total_loss(model, (XF[tr[j]] - model.mf) / model.sf, (XV[tr[j]] - model.mv) / model.sv, ytr[j], cfg)
            opt.zero_grad(); loss.backward(); opt.step()
            if ema is not None:
                with torch.no_grad():
                    for pe, pm in zip(ema.state_dict().values(), model.state_dict().values()):
                        if pe.dtype.is_floating_point:
                            pe.mul_(0.995).add_(pm, alpha=0.005)
                        else:
                            pe.copy_(pm)
        evm = ema if ema is not None else model
        if vsets is None:
            continue
        sc = val_scores(evm, XF, XV, vsets)
        score = np.mean([eer(L, s) for (Vi, Fi, L), s in zip(vsets, sc)])
        if score < best - 0.05:
            best, best_ep, best_state, wait = score, ep, copy.deepcopy(evm.state_dict()), 0
        else:
            wait += 1
            if wait >= PATIENCE:
                break
    final = ema if ema is not None else model
    if best_state is not None:
        final.load_state_dict(best_state)
    return final, best_ep, XF, XV, ch_stats

VARIANTS = {
    'H1_fop_indrop': dict(arch='fop', in_drop=0.5),
    'H2_fop_ema':    dict(arch='fop', ema=True),
    'H3_fop_nce':    dict(arch='fop', nce=0.5),
    'H4_lin_mse':    dict(arch='lin', d=192, in_drop=0.7, mse=1.0),
    'H5_lin_nce':    dict(arch='lin', d=192, in_drop=0.7, nce=0.5),
}
CSV = os.path.join(RESDIR, 'nb11_cv.csv')
done = pd.read_csv(CSV) if os.path.exists(CSV) else pd.DataFrame(columns=['variant', 'seed', 'fold'])
t0 = time.time()
for name, cfg in VARIANTS.items():
    for seed in SEEDS:
        for k in range(K):
            if ((done.variant == name) & (done.seed == seed) & (done.fold == k)).any():
                continue
            tr_idx, vsets = FOLD_DATA[k]
            model, best_ep, XF, XV, _ = train_variant(tr_idx, cfg, vsets, seed=seed)
            sc = val_scores(model, XF, XV, vsets)
            np.savez(os.path.join(SCOREDIR, '%s_s%d_f%d.npz' % (name, seed, k)), std=sc[0], gen=sc[1])
            e_std, e_gen = [eer(L, s) for (Vi, Fi, L), s in zip(vsets, sc)]
            row = pd.DataFrame([dict(variant=name, seed=seed, fold=k, best_ep=best_ep, std=e_std, gender=e_gen)])
            row.to_csv(CSV, mode='a', header=not os.path.exists(CSV), index=False)
            done = pd.concat([done, row])
            print('%-14s seed %d fold %d | ep %3d | std %.2f | gender %.2f | %.0f min'
                  % (name, seed, k, best_ep, e_std, e_gen, (time.time() - t0) / 60))

In [ ]:
# Cell 6 - results table + ENSEMBLE test (average of z-scored validation scores; z-stats from the same validation fold)
N_VOXP = len(set(VOX['spk']))
REF = 'E3_vox%d_dn_bal75' % N_VOXP
R8 = pd.read_csv(os.path.join(RESDIR, 'nb08_cv.csv')); R8 = R8[R8.variant == REF]
R = pd.concat([R8, pd.read_csv(CSV)], ignore_index=True)
R['overall'] = (R['std'] + R['gender']) / 2
per_seed = R.groupby(['variant', 'seed'])[['std', 'gender', 'overall']].mean().reset_index()
tab = per_seed.groupby('variant').agg(std=('std', 'mean'), gender=('gender', 'mean'),
                                      overall=('overall', 'mean'), seed_sd=('overall', 'std'))
tab = tab.loc[[v for v in [REF] + list(VARIANTS) if v in tab.index]].round(2)
note('HEAD & TRAINING TRICKS (validation on held-out CHALLENGE people, 5-fold x 3 seeds; single models)')
note(tab.to_string())

def ens_eer(members):
    # per fold: average z-scored scores over members x seeds, then EER; returns (std, gender, overall)
    res = []
    for k in range(K):
        _, vsets = FOLD_DATA[k]
        out = []
        for j, key in enumerate(['std', 'gen']):
            acc = 0
            for m in members:
                for s in SEEDS:
                    a = np.load(os.path.join(SCOREDIR, '%s_s%d_f%d.npz' % (m, s, k)))[key]
                    acc = acc + (a - a.mean()) / (a.std() + 1e-9)
            out.append(eer(vsets[j][2], acc))
        res.append(out)
    r = np.array(res).mean(0)
    return r[0], r[1], r.mean()

rows = []
for m in VARIANTS:
    rows.append(('seeds-only: ' + m,) + ens_eer([m]))
single_best = tab.loc[list(VARIANTS)]['overall'].sort_values().index.tolist()
for n in [2, 3, len(VARIANTS)]:
    rows.append(('top-%d versions: %s' % (n, '+'.join(single_best[:n])),) + ens_eer(single_best[:n]))
E = pd.DataFrame(rows, columns=['ensemble', 'std', 'gender', 'overall']).round(2)
note('ENSEMBLES (3 seeds per version always averaged, like our final submissions)')
note(E.to_string(index=False))

In [ ]:
# Cell 7 - submission ZIP for the best single NEW version (if it beats the reference)
LOWER_IS_SAME = True
LANG_FILE = {'English': 'sub_score_v4_English_heard.txt', 'Bangla': 'sub_score_v4_Bangla_unheard.txt'}
BEST = tab.loc[list(VARIANTS)]['overall'].idxmin()
if tab.loc[BEST, 'overall'] < tab.loc[REF, 'overall']:
    cfg = VARIANTS[BEST]
    ep = int(np.median(R[R.variant == BEST]['best_ep']))
    finals = []
    for s in SEEDS:
        m, _, _, _, ch_stats = train_variant(np.arange(N_CH), cfg, None, seed=s, max_epochs=ep)
        finals.append(m.eval())
    OUT = '/content/sub_11_' + BEST
    for track in ['no_gender', 'gender']:
        os.makedirs(os.path.join(OUT, track), exist_ok=True)
        for lang, fname in LANG_FILE.items():
            split = track + '_' + lang
            pair_ids = [l.split()[0] for l in open(find_one('/' + track + '/', lang + '_test.txt')) if l.strip()]
            fx = ((face_of(FEAT['clip'][split]) - ch_stats[0]) / ch_stats[1]).astype(np.float32)        # challenge TRAIN stats
            vx = ((voice_of(FEAT['ecapa'][split], FEAT['wavlm_sv'][split]) - ch_stats[2]) / ch_stats[3]).astype(np.float32)
            xf = torch.tensor(fx, device=dev); xv = torch.tensor(vx, device=dev)
            with torch.no_grad():
                s_ = np.mean([m.score((xf - m.mf) / m.sf, (xv - m.mv) / m.sv).cpu().numpy() for m in finals], axis=0)
            out = -s_ if LOWER_IS_SAME else s_
            with open(os.path.join(OUT, track, fname), 'w') as f:
                for pid, v in zip(pair_ids, out):
                    f.write('%s %.6f\n' % (pid, v))
    SUBDIR = os.path.join(PROJECT, 'submissions'); os.makedirs(SUBDIR, exist_ok=True)
    ZIP_PATH = os.path.join(SUBDIR, '11_' + BEST + '.zip')
    with zipfile.ZipFile(ZIP_PATH, 'w', zipfile.ZIP_DEFLATED) as zf:
        for track in ['no_gender', 'gender']:
            for fname in LANG_FILE.values():
                zf.write(os.path.join(OUT, track, fname), arcname=track + '/' + fname)
    note('Submission ZIP saved: %s (%d epochs)' % (ZIP_PATH[len(BASE):], ep))
else:
    note('No single new version beats the reference - no ZIP built (see ensemble table).')

LOG = os.path.join(PROJECT, 'experiments.csv')
log = pd.read_csv(LOG) if os.path.exists(LOG) else pd.DataFrame()
if len(log) and 'run' in log:
    log = log[~log.run.isin(['11_' + v for v in VARIANTS])]
new = pd.DataFrame([{'date': time.strftime('%Y-%m-%d %H:%M'), 'run': '11_' + v, 'change': 'bal75 data + head cfg %s' % VARIANTS[v],
                     'seed': '1,2,3', 'cv_std': tab.loc[v, 'std'], 'cv_gender': tab.loc[v, 'gender'],
                     'cv_overall': tab.loc[v, 'overall'], 'codabench_dev': ''} for v in VARIANTS if v in tab.index])
pd.concat([log, new], ignore_index=True).to_csv(LOG, index=False)
print('=' * 70)
print('SUMMARY  (copy everything between the lines and paste to Claude)')
print('=' * 70)
for s in SUMMARY:
    print(s)
print('=' * 70)